# demo

> Interactive demos on the `learning` shelf: store them, make them in the background after repeated misses, and log each prediction.

A demo is one interactive figure from [rupaka](https://github.com/vedicreader/rupaka). It asks the learner to predict first, and shows the result after the lock in.
This module keeps demos on the `learning` shelf, as concepts and attempts are. It decides when the tutor makes a demo, and it runs that job in the background, so the learner never waits.

| rule | what happens |
|---|---|
| a miss on a concept that has a demo | the result of the question gets the demo, and the session shows it under the lesson |
| two misses with a saved misconception, or a concept with `visual: true` in its meta | the tutor queues a job: a model writes a demo, and rupaka checks it |
| the learner asks ("Show me a demo" on the concept page) | the same job, at once |
| a prediction in a demo | an attempt with `kind='demo'`; the learner model does not change |

In [ ]:
#| default_exp demo

In [ ]:
#| hide
from fastcore.test import *

In [ ]:
#| hide
import os, tempfile
os.environ['VISHALAKSHI_VAULT'] = os.environ['ABHYASA_VAULT'] = os.path.join(tempfile.mkdtemp(), 'vault.db')   # a test that opens the default vault gets a temporary one

In [ ]:
#| export
import json, time
from concurrent.futures import ThreadPoolExecutor
from fastcore.utils import *
import rupaka
from rupaka.spec import validate, upgrade, gallery
from rupaka.agent import author
from abhyasa.store import Learning, slugify, _src
from abhyasa.tutor import CLOSE_HOOKS

## Storage

A demo is one vault document. Its `source` is `learning://demo/<slug>`, its `kind` is `demo`, and its meta holds the spec.

| meta | value |
|---|---|
| `concept` | the slug of the concept, the link |
| `spec` | the rupaka spec |
| `schema`, `rupaka` | the schema of the spec, and the rupaka version that checked it |
| `by` | `model:<name>` or `hand` |
| `checked` | the time of the last passing check, or None |
| `stats` | `shown`: predictions made; `right`: predictions that were right |

The text of the document is the title, the question, the gate prompts and the captions, so hybrid search finds a demo.

In [ ]:
#| export
def demo_text(spec) -> str:
    "The searchable text of a demo: title, ask, gate prompts and captions."
    def walk(bs):
        for b in bs or []:
            yield from (b[k] for k in ('prompt', 'text') if isinstance(b.get(k), str))
            yield from walk(b.get('blocks'))
    return '\n\n'.join(t for t in (spec.get('title'), spec.get('ask'), *walk(spec.get('blocks'))) if t)

`put_demo` adds a demo or replaces one with the same slug. It keeps the stats of the old one. The vault keeps one document for one text, so the text ends with the slug and the concept: two demos with the same spec stay two documents.

In [ ]:
#| export
@patch
def put_demo(self:Learning, spec:dict, concept:str, by:str='hand', checked:float=None) -> AttrDict:
    "Store `spec` as a demo of `concept`."
    spec = upgrade(spec)
    slug = slugify(spec['id'])
    old = self.demo(slug) or {}
    m = dict(slug=slug, concept=concept, spec=spec, schema=spec['rupaka'], rupaka=rupaka.__version__, by=by,
             checked=checked, at=time.time(), stats=old.get('stats') or dict(shown=0, right=0))
    r = self.v.add(f'{demo_text(spec)}\n\nDemo {slug} for {concept}', title=spec.get('title') or slug, source=_src('demo', slug), kind='demo', meta=m)
    if not r.get('changed'): self.v.set_meta(r['doc_id'], **m)
    return self.demo(slug)

`demo` gets one demo by slug. `demos` lists the demos of a concept, newest first. `demo_for` gives the newest one.

In [ ]:
#| export
def _meta(r): return AttrDict({**json.loads(r['meta'] or '{}'), 'doc_id': r['id']})

@patch
def demo(self:Learning, slug:str) -> AttrDict|None:
    "One demo by slug."
    r = first(self.v.t.docs(where='source=?', where_args=[_src('demo', slug)]))
    return _meta(r) if r else None

@patch
def demos(self:Learning, concept:str=None) -> L:
    "The demos of `concept` (or all demos), newest first."
    rs = self._docs('demo', *(("json_extract(meta, '$.concept')=?", [concept]) if concept else ()))
    return L(sorted(map(_meta, rs), key=lambda d: -d.get('at', 0)))

@patch
def demo_for(self:Learning, concept:str) -> AttrDict|None:
    "The newest demo of `concept`, or None."
    return first(self.demos(concept))

### Tests for storage

A shelf in a temporary vault, with one concept and the sandhi demo from the rupaka gallery.

In [ ]:
from abhyasa.store import temp_learning
lr = temp_learning()
lr.put('Vowel sandhi', 'Two vowels at a word boundary join by rule.', 'sanskrit', slug='sa-sandhi-vowels')
spec = gallery()['vowel-sandhi']
d = lr.put_demo(spec, 'sa-sandhi-vowels')
test_eq((d.slug, d.concept, d.by, d.stats), ('vowel-sandhi', 'sa-sandhi-vowels', 'hand', dict(shown=0, right=0)))

The demo is found by its concept, and by a search for the words of its caption.

In [ ]:
test_eq(lr.demo_for('sa-sandhi-vowels').spec, spec)
test_eq(lr.demos('nope'), [])
test_eq(lr.search('rule stops another starts', kind='demo')[0]['doc_id'], d.doc_id)

A demo is not a concept, so the list of concepts stays the same.

In [ ]:
test_eq(lr.concepts().attrgot('slug'), ['sa-sandhi-vowels'])

## Predictions

The demo page sends each prediction. `record_prediction` files it as an attempt with `kind='demo'`, with the gate, the pick, the right answer and the "why" line.
It counts the prediction in the stats of the demo. It does not change the learner model: a demo is practice, and the next question still tests the concept.

In [ ]:
#| export
def _gate(spec, gid):
    def walk(bs):
        for b in bs or []:
            if b.get('type') == 'gate' and b.get('id') == gid: return b
            if (g := walk(b.get('blocks'))): return g
    return walk(spec.get('blocks')) or {}

@patch
def record_prediction(self:Learning, slug:str, gate:str, picked, ans, ok:bool, why:str='', context:str='', session:str='') -> AttrDict:
    "File one prediction in demo `slug` as an attempt, and count it in the stats of the demo."
    d = self.demo(slug)
    if d is None: raise KeyError(f'no demo {slug!r}')
    st = d.get('stats') or {}
    self.v.set_meta(d.doc_id, stats=dict(shown=st.get('shown', 0) + 1, right=st.get('right', 0) + bool(ok)))
    q = f"{context} {_gate(d.spec, gate).get('prompt', gate)}".strip()
    return self.record(d.concept, q, str(picked), why, bool(ok), kind='demo', depth=1, session=session, final=True,
                       feedback=f'The answer is {ans}.', demo=slug, gate=gate, expected=str(ans))

In [ ]:
c0 = lr.concept('sa-sandhi-vowels')
a = lr.record_prediction('vowel-sandhi', 'join', 'ai', 'e', False, why='a and i make a diphthong', context='For a + i:', session='s1')
test_eq((a.kind, a.correct, a.thinking, a.demo, a.expected), ('demo', False, 'a and i make a diphthong', 'vowel-sandhi', 'e'))
test_eq(a.question, 'For a + i: What do the two vowels become?')
test_eq(lr.demo('vowel-sandhi').stats, dict(shown=1, right=0))

The learner model of the concept does not change.

In [ ]:
c1 = lr.concept('sa-sandhi-vowels')
test_eq({k: c1[k] for k in ('floor', 'misses', 'seen')}, {k: c0[k] for k in ('floor', 'misses', 'seen')})

## When to make a demo

`want_demo` says if a concept should get a demo: after `DEMO_MISSES` misses with a saved misconception, or when its meta has `visual: true`.

In [ ]:
#| export
DEMO_MISSES = 2

def want_demo(c) -> bool:
    "True when concept `c` should get a demo."
    return bool(c and (c.get('visual') or (c.get('misses', 0) >= DEMO_MISSES and c.get('misconceptions'))))

In [ ]:
assert not want_demo(dict(misses=1, misconceptions=['x'])) and not want_demo(dict(misses=3, misconceptions=[]))
assert want_demo(dict(misses=2, misconceptions=['x'])) and want_demo(dict(visual=True))

## Make a demo

`make_demo` asks the model for a demo with `rupaka.author`, and stores it. The loop validates the spec, checks it in headless Chrome when a browser is there, and repairs it up to two times.
When the model cannot make a good demo, `make_demo` returns None and stores nothing. The text lesson stays.

In [ ]:
#| export
def make_demo(learning:Learning, ask, slug:str, first_error:str='', check=None, log:list=None) -> AttrDict|None:
    "A new checked demo for concept `slug`, stored on the shelf; None when the model fails."
    c = learning.concept(slug)
    if c is None: return None
    if check is None:
        from rupaka.check import can_check
        check = can_check()
    spec = author(c, ask, first_error, check=check, log=log)
    if spec is None: return None
    return learning.put_demo(spec, slug, by=f"model:{getattr(ask, 'model', 'unknown')}", checked=time.time() if check else None)

A fake model plays the agent. It replies with a demo for the concept, or with text that is not JSON.

In [ ]:
from abhyasa.llm import Fake
good = {**spec, 'id': 'sandhi-by-model'}
fk = Fake({'You design one interactive demo': good})
d = make_demo(lr, fk, 'sa-sandhi-vowels', check=False)
test_eq((d.slug, d.by, d.checked), ('sandhi-by-model', 'model:unknown', None))
test_eq(lr.demo_for('sa-sandhi-vowels').slug, 'sandhi-by-model')

A model that never gives a good spec stores nothing, and the log names the errors.

In [ ]:
log = []
test_eq(make_demo(lr, Fake({'You design one interactive demo': 'no demo today'}), 'sa-sandhi-vowels', check=False, log=log), None)
test_eq(len(log), 3)
test_eq(len(lr.demos('sa-sandhi-vowels')), 2)

## The background job

`queue_demo` runs `make_demo` in the thread pool of the tutor. It runs one job at a time for each concept: a second call while the job runs gives the same job.

In [ ]:
#| export
_POOL = ThreadPoolExecutor(1)

def queue_demo(tutor, slug:str, first_error:str=''):
    "Start a background job that makes a demo for `slug`; the running job if there is one."
    j = tutor.jobs.get(slug)
    if j is not None and not j.done(): return j
    j = tutor.jobs[slug] = (tutor.pool or _POOL).submit(make_demo, tutor.learning, tutor.ask, slug, first_error)
    return j

def job_state(tutor, slug:str) -> str:
    "'running', 'failed', 'done' or 'none' for the demo job of `slug`."
    j = tutor.jobs.get(slug)
    if j is None: return 'none'
    if not j.done(): return 'running'
    return 'done' if not j.exception() and j.result() else 'failed'

## The tutor hook

`on_close` runs after each question closes (it is in `CLOSE_HOOKS`). After a miss, it gives the result the newest demo of the concept.
When there is no demo yet and the concept wants one, it queues the job and marks the result with `demo_pending`. A pass adds nothing.

In [ ]:
#| export
def on_close(tutor, q, r):
    "After a miss: attach the newest demo of the concept to the result, or queue a job when the concept wants a demo."
    if r.correct: return r
    if (d := tutor.learning.demo_for(q.concept)): r.demo = d.slug
    elif want_demo(tutor.learning.concept(q.concept)): queue_demo(tutor, q.concept, r.get('first_error', '')); r.demo_pending = True
    return r

CLOSE_HOOKS[:] = [f for f in CLOSE_HOOKS if getattr(f, '__name__', '') != 'on_close'] + [on_close]

### Tests for the hook

A session where each answer is a sure miss with the same misconception. The fake model writes questions, grades, lessons and demos.

In [ ]:
from abhyasa.tutor import Tutor
from concurrent.futures import ThreadPoolExecutor
def grader(p, sp): return dict(correct=False, grade=1, first_error='You add the vowels.', misconception='vowels just add', feedback='Look at the rule.')
fk = Fake({'You grade one answer': grader, 'Teach the smallest step': 'One step. Check: what is a + u?',
           'You design one interactive demo': good}, dflt=dict(question='What is a + i?', answer='e', rubric=['e']))
lr2 = temp_learning()
lr2.put('Vowel sandhi', 'Two vowels join by rule.', 'sanskrit', slug='sa-sandhi-vowels')
t = Tutor(lr2, fk, context=False, pool=ThreadPoolExecutor(1))
def miss():
    q = t.next(focus='sa-sandhi-vowels')
    return t.give_up(q.id, confidence=90)

The first miss queues nothing: there is one miss.

In [ ]:
r1 = miss()
test_eq((r1.get('demo'), r1.get('demo_pending'), lr2.concept('sa-sandhi-vowels').misses), (None, None, 1))

A miss with a misconception saves the misconception. The second such miss queues one job, and a third call while it runs does not queue another.

In [ ]:
lr2.update('sa-sandhi-vowels', misconceptions=['vowels just add'])
r2 = miss()
j = t.jobs['sa-sandhi-vowels']
assert r2.demo_pending and (queue_demo(t, 'sa-sandhi-vowels') is j or j.done())
t.jobs['sa-sandhi-vowels'].result()
test_eq(job_state(t, 'sa-sandhi-vowels'), 'done')

Now the concept has a demo, so the next miss gets its slug in the result.

In [ ]:
r3 = miss()
test_eq(r3.demo, 'sandhi-by-model')

A pass gets no demo.

In [ ]:
q = t.next(focus='sa-sandhi-vowels')
r = t._close(t.open[q.id], True, 80, AttrDict(correct=True, grade=9))
assert 'demo' not in r

## The gallery

`import_gallery` stores the gallery demos of rupaka for the concepts they teach. A demo goes to the concept whose slug, or whose slugified name, is the concept of the demo or its title.
`links` maps a gallery id to a concept slug, for other cases.

In [ ]:
#| export
def import_gallery(learning:Learning, links:dict=None) -> L:
    "Store the rupaka gallery demos for the matching concepts; return the stored demos."
    links, out = links or {}, L()
    cs = learning.concepts()
    for gid, s in gallery().items():
        keys = {s.get('concept'), slugify(s.get('title', '')), gid}
        slug = links.get(gid) or first(c.slug for c in cs if c.slug in keys or slugify(c.name) in keys)
        if slug: out.append(learning.put_demo(s, slug))
    return out

In [ ]:
test_eq(import_gallery(lr2).attrgot('slug'), ['vowel-sandhi'])
test_eq(import_gallery(lr2, links={'nakshatra-pada': 'sa-sandhi-vowels'}).attrgot('slug'), ['nakshatra-pada', 'vowel-sandhi'])

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()